> **Solución.** Challenge de ML (pipeline de clasificación de vinos) con los `TODO` completados y las preguntas respondidas. Los tres notebooks se ejecutan **en orden** (1 → 2 → 3) y comparten artefactos en `data/`, `artifacts/` y `reports/`.
>
> Nicolás Rodríguez

# Challenge 3: evaluación final e inferencia

El modelo ya fue seleccionado. Ahora debes abrir el conjunto de test una sola vez, comunicar su desempeño y comprobar que el pipeline puede reutilizarse con datos nuevos.

**Entradas:** `test.csv`, `data_contract.json` y `champion_model.joblib`  
**Salidas:** `test_metrics.json`, `batch_predictions.csv`

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    f1_score,
)

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga del modelo y del conjunto de prueba

In [3]:
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))
training_metadata = json.loads((ARTIFACTS_DIR / "training_metadata.json").read_text(encoding="utf-8"))
model = joblib.load(ARTIFACTS_DIR / "champion_model.joblib")

test_df = pd.read_csv(PROCESSED_DATA_DIR / "test.csv")
X_test = test_df[contract["features"]]
y_test = test_df[contract["target"]]

print("Modelo:", training_metadata["champion"])
print("Test:", X_test.shape)

Modelo: LogisticRegression
Test: (36, 13)


## 2. Evaluación final

In [4]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro")
print(f"Accuracy test : {acc:.4f}")
print(f"F1 macro test : {f1_macro:.4f}")
print("\n", classification_report(y_test, y_pred))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred)
plt.title("Matriz de confusión — test"); plt.show()

Accuracy test : 0.9722
F1 macro test : 0.9710

               precision    recall  f1-score   support

           0       1.00      1.00      1.00        12
           1       0.93      1.00      0.97        14
           2       1.00      0.90      0.95        10

    accuracy                           0.97        36
   macro avg       0.98      0.97      0.97        36
weighted avg       0.97      0.97      0.97        36



/var/folders/wm/z3k3lhg50gl4qtr99hh4f09r0000gn/T/ipykernel_34265/2948977781.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.title("Matriz de confusión — test"); plt.show()


**Análisis:**

**Análisis.** El accuracy de test queda muy cerca del de CV (diferencia de pocos puntos), lo que indica que **no hay sobreajuste** y que el split fue representativo. En la matriz de confusión, si hay errores suelen ser entre las dos clases con química más parecida (clases 1 y 2). Con 36 filas de test, un solo error mueve el accuracy ~3 %, así que conviene reportar también el F1 macro.

## 3. Función de inferencia

In [5]:
feature_columns = contract["features"]


def predict(records: pd.DataFrame) -> pd.DataFrame:
    """Valida el esquema y devuelve las predicciones del pipeline."""
    faltantes = [c for c in feature_columns if c not in records.columns]
    if faltantes:
        raise ValueError(f"Faltan columnas: {faltantes}")
    X = records[feature_columns].copy()          # ordena las columnas
    proba = model.predict_proba(X)
    out = pd.DataFrame(index=records.index)
    out["prediction"] = model.predict(X)
    for i, cls in enumerate(model.classes_):
        out[f"proba_{cls}"] = proba[:, i]
    return out

## 4. Predicción por lotes

In [6]:
# Simulación de datos nuevos. En producción no provendrían del test set.
new_records = X_test.iloc[:5].copy()

batch_predictions = predict(new_records)
batch_predictions.to_csv(REPORTS_DIR / "batch_predictions.csv", index=False)
batch_predictions

,prediction,proba_0,proba_1,proba_2
0,0,0.999671,0.000307,0.000023
1,1,0.007051,0.540423,0.452526
2,0,0.992000,0.007664,0.000336
3,1,0.289967,0.707467,0.002565
4,1,0.033032,0.962313,0.004655


## 5. Reporte final

In [7]:
test_metrics = {
    "champion": training_metadata["champion"],
    "accuracy": float(acc),
    "f1_macro": float(f1_macro),
    "cv_accuracy": training_metadata["cv_accuracy"],
}
(REPORTS_DIR / "test_metrics.json").write_text(
    json.dumps(test_metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

{
  "champion": "LogisticRegression",
  "accuracy": 0.9722222222222222,
  "f1_macro": 0.9709618874773139,
  "cv_accuracy": 0.9790640394088669
}


In [8]:
assert (REPORTS_DIR / "test_metrics.json").exists()
assert (REPORTS_DIR / "batch_predictions.csv").exists()
print("Challenge 3 completado.")

Challenge 3 completado.


## Resumen ejecutivo

| Elemento | Resultado |
|---|---|
| Modelo campeón | ver `artifacts/training_metadata.json` |
| Accuracy CV | ver `cv_accuracy` en `training_metadata.json` |
| Accuracy test | ver `reports/test_metrics.json` |
| F1 macro test | ver `reports/test_metrics.json` |
| Clases más confundidas | 1 y 2 (química más parecida) |
| Principal limitación | test pequeño (36 filas): cada error pesa ~3 % |

## Checklist

- [ ] Evalué únicamente el modelo campeón.
- [ ] No reajusté hiperparámetros después de observar test.
- [ ] Implementé validación básica del esquema de entrada.
- [ ] Generé predicciones por lotes.
- [ ] Guardé las métricas finales.